In [0]:
from pyspark.sql.functions import col, when
from pyspark.sql.types import DoubleType

df = spark.table("weather_project.bronze.daily_historical")

# has_rain: переводим boolen-like string в boolean 

df = df.withColumn("has_rain", when(col("has_rain") == "true", True).when(col("has_rain") == "false", False).otherwise(None))

df = df.withColumn("snow_total",col("snow_total").cast(DoubleType()))

df = df.withColumn("minutes_of_ice_total",col("minutes_of_ice_total").cast(DoubleType()))


# Corrupted records — в данном датасете все три колонки snow_total, has_rain, minutes_of_ice_total NULL в источнике,
# поэтому фильтр по ним удалит все строки. Пропускаем.

df.write \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .saveAsTable("weather_project.silver.daily_clean")

# Считаем количество строк

daily_clean_count = df.count()
print(f" Silver daily: {daily_clean_count} rows")

# Сохраняем значение daily_clean_count  под ключом "daily_clean_count, чтобы потом вызвать в 04_quality_checks

dbutils.jobs.taskValues.set(key="daily_clean_count", value=daily_clean_count)

display(spark.read.table("weather_project.silver.daily_clean").limit(5))